# Entity Sentiment Model - Longformer-Large Training (A100 80GB)

Fresh two-stage training with **Longformer-Large** on cleaned data.

**Model:**
- Encoder: `allenai/longformer-large-4096` (24 layers, 1024-dim, ~435M params)
- NER Head: CRF-based token classification (15 BIO labels)
- Sentiment Head: Attention-based regression to [-1, 1]

**Data (cleaned 2025-02-11):**
- Training: `data/labeled/final/train.jsonl` (17,593 articles)
- Validation: `data/labeled/final/val.jsonl` (2,130 articles)
- Holdout: `data/labeled/final/holdout.jsonl` (9,371 articles)

**Hardware & Batch Size:**
- GPU: A100 80GB
- `batch_size=4`, `gradient_accumulation=3` -> effective batch = 12
- Mixed precision (AMP) enabled

**Memory Budget (Longformer-Large, seq_len=2048):**

| Component | Estimate |
|-----------|----------|
| Model params (FP32) | ~1.7 GB |
| Optimizer states (AdamW) | ~3.5 GB |
| Gradients | ~1.7 GB |
| Activations (batch=4) | ~63 GB |
| **Total** | **~70 GB** |
| Headroom | ~10 GB |

*Scaled from base model run (73 GB peak at batch=12): activations scale ~2.67x for large (24 vs 12 layers, 1024 vs 768 hidden).*

## 1. Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set project path
import os

PROJECT_PATH = "/content/drive/MyDrive/entity_sentiment_model_pipeline"

assert os.path.exists(PROJECT_PATH), f"Project path not found: {PROJECT_PATH}\nRun: !ls -la /content/drive/ to see available paths"
print(f"Project found at: {PROJECT_PATH}")

# List contents
!ls -la "{PROJECT_PATH}"

In [ ]:
# Install dependencies
!pip install -q transformers torch torchvision torchaudio
!pip install -q pytorch-crf
!pip install -q accelerate

In [ ]:
# Authenticate with Hugging Face Hub (for faster downloads + no warnings)
#
# OPTION A (recommended): Use Colab Secrets
#   1. Go to https://huggingface.co/settings/tokens and create a read-only token
#   2. In Colab: click the key icon (left sidebar) -> Add new secret
#      Name: HF_TOKEN, Value: your token, toggle "Notebook access" on
#   3. Restart runtime, then this cell will auto-authenticate
#
# OPTION B: Paste token when prompted (if Secrets not configured)

import os
from getpass import getpass

hf_token = None

# Try Colab Secrets first
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print("Found HF_TOKEN in Colab Secrets")
except (ImportError, userdata.SecretNotFoundError):
    print("HF_TOKEN not found in Colab Secrets.")
    hf_token = getpass("Paste your HF token (or press Enter to skip): ").strip()
    if not hf_token:
        print("Skipping HF auth — downloads will still work for public models, just slower")

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print("Authenticated with Hugging Face Hub")

In [ ]:
# Add project to Python path
import sys
sys.path.insert(0, PROJECT_PATH)

# Change to project directory
os.chdir(PROJECT_PATH)
print(f"Working directory: {os.getcwd()}")

## 2. Training Overview & Data

In [ ]:
# Fresh training with Longformer-Large
# No previous checkpoint — training both stages from scratch

print("=" * 60)
print("FRESH TRAINING: Longformer-Large (1024-dim)")
print("=" * 60)
print()
print("Model: allenai/longformer-large-4096")
print("  - 24 transformer layers (vs 12 in base)")
print("  - 1024 hidden dimensions (vs 768 in base)")
print("  - ~435M parameters (vs ~149M in base)")
print()
print("Training plan:")
print("  Stage 1: NER-only training (5 epochs)")
print("  Stage 2: Joint NER+Sentiment fine-tuning (5 epochs)")

In [ ]:
# Training data — cleaned 2025-02-11
# Removed: corrupted articles, invalid entity types, empty entities
# Fixed: empty ner_mentions positions, PERSON null sentiment
TRAIN_FILE = f"{PROJECT_PATH}/data/labeled/final/train.jsonl"
VAL_FILE = f"{PROJECT_PATH}/data/labeled/final/val.jsonl"
HOLDOUT_FILE = f"{PROJECT_PATH}/data/labeled/final/holdout.jsonl"

print("Data files:")
for label, path in [("Train", TRAIN_FILE), ("Val", VAL_FILE), ("Holdout", HOLDOUT_FILE)]:
    print(f"  {label}: {path}")
print()
!wc -l "{TRAIN_FILE}" "{VAL_FILE}" "{HOLDOUT_FILE}"

## 3. Configuration (A100 80GB Optimized)

In [ ]:
from pathlib import Path
from datetime import datetime

# Training configuration — Longformer-Large on A100 80GB
CONFIG = {
    # Data — cleaned 2025-02-11
    "train_file": TRAIN_FILE,
    "val_file": VAL_FILE,
    "holdout_file": HOLDOUT_FILE,

    # Model — LARGE
    "encoder_name": "allenai/longformer-large-4096",
    "hidden_size": 1024,
    "max_length": 2048,
    "use_crf": True,

    # A100 80GB memory budget (measured from Stage 1):
    #   Observed: 52.9 GB at batch_size=4 → ~11.5 GB/sample activations
    #   batch_size=6: 6 * 11.5 + 6.9 ≈ 76 GB (4 GB headroom)
    #   Stage 2 uses ~5-10 GB more (sentiment head) — still fits
    "batch_size": 5,
    "gradient_accumulation": 2,  # effective batch = 12

    # Stage 1: NER-only pretraining
    "stage1_epochs": 5,
    "stage1_lr": 2e-5,

    # Stage 2: Joint fine-tuning
    "stage2_epochs": 5,
    "stage2_lr": 1e-5,

    # Checkpoints — separate from base model
    "stage1_checkpoint_dir": f"{PROJECT_PATH}/checkpoints/stage1_ner_large",
    "stage2_checkpoint_dir": f"{PROJECT_PATH}/checkpoints/stage2_joint_large",
}

# Create checkpoint directories
os.makedirs(CONFIG["stage1_checkpoint_dir"], exist_ok=True)
os.makedirs(CONFIG["stage2_checkpoint_dir"], exist_ok=True)

print("Configuration (Longformer-Large, A100 80GB):")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")
print(f"\nEffective batch size: {CONFIG['batch_size'] * CONFIG['gradient_accumulation']}")
print(f"Batches per epoch: ~{17593 // CONFIG['batch_size']}")
print(f"Optimizer steps per epoch: ~{17593 // (CONFIG['batch_size'] * CONFIG['gradient_accumulation'])}")

In [ ]:
import torch
from training.preprocessing import DataPreprocessor
from training.dataset import create_data_loaders
from models.pipeline import FinancialEntitySentimentModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


## 4. Load Data

In [ ]:
# Initialize preprocessor with large model tokenizer
preprocessor = DataPreprocessor(
    model_name=CONFIG["encoder_name"],
    max_length=CONFIG["max_length"],
)

print("Loading and preprocessing data (this may take a few minutes)...")
train_loader, val_loader = create_data_loaders(
    train_files=CONFIG["train_file"],
    val_files=CONFIG["val_file"],
    preprocessor=preprocessor,
    batch_size=CONFIG["batch_size"],
)

print(f"\nData loaded:")
print(f"  Train batches: {len(train_loader)} (batch_size={CONFIG['batch_size']})")
print(f"  Val batches: {len(val_loader)}")
print(f"  Train samples: ~{len(train_loader) * CONFIG['batch_size']}")
print(f"  Val samples: ~{len(val_loader) * CONFIG['batch_size']}")

## 5. Custom Training Loop with Gradient Accumulation

Since the trainer doesn't natively support gradient accumulation, we implement a custom training loop.

In [ ]:
import time
from tqdm.auto import tqdm
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from training.trainer import compute_ner_metrics, compute_sentiment_metrics

def train_epoch_with_accumulation(
    model,
    train_loader,
    optimizer,
    scheduler,
    scaler,
    device,
    accumulation_steps,
    ner_weight=1.0,
    sentiment_weight=0.0,
    gradient_clip=1.0,
):
    """Train for one epoch with gradient accumulation."""
    model.train()

    total_loss = 0.0
    total_ner_loss = 0.0
    total_sentiment_loss = 0.0
    num_batches = 0

    ner_criterion = nn.CrossEntropyLoss(label_smoothing=0.1, ignore_index=-100)
    sentiment_criterion = nn.MSELoss(reduction="none")

    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc="Training")
    for step, batch in enumerate(pbar):
        # Move to device
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        ner_labels = batch["ner_labels"].to(device)
        entity_masks = batch["entity_masks"].to(device)
        sentiment_targets = batch["sentiment_scores"].to(device)
        entity_mask_valid = batch["entity_mask_valid"].to(device)

        # Forward with mixed precision
        with torch.amp.autocast(device_type="cuda"):
            ner_output, sentiment_preds = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                entity_masks=entity_masks,
                ner_labels=ner_labels,
            )

            # NER loss
            if isinstance(ner_output, dict):
                ner_loss = ner_output.get("loss", torch.tensor(0.0, device=device))
            else:
                batch_size, seq_len, num_labels = ner_output.shape
                ner_loss = ner_criterion(
                    ner_output.view(-1, num_labels),
                    ner_labels.view(-1)
                )

            # Sentiment loss
            sentiment_loss_all = sentiment_criterion(sentiment_preds, sentiment_targets)
            sentiment_loss_masked = sentiment_loss_all * entity_mask_valid
            num_valid = entity_mask_valid.sum()
            if num_valid > 0:
                sentiment_loss = sentiment_loss_masked.sum() / num_valid
            else:
                sentiment_loss = torch.tensor(0.0, device=device)

            # Combined loss (scaled for accumulation)
            loss = (ner_weight * ner_loss + sentiment_weight * sentiment_loss) / accumulation_steps

        # Backward with scaling
        scaler.scale(loss).backward()

        # Accumulate metrics
        total_loss += loss.item() * accumulation_steps
        total_ner_loss += ner_loss.item()
        total_sentiment_loss += sentiment_loss.item() if num_valid > 0 else 0
        num_batches += 1

        # Optimizer step every accumulation_steps
        if (step + 1) % accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

            # Update progress bar
            pbar.set_postfix({
                "loss": f"{total_loss/num_batches:.4f}",
                "ner": f"{total_ner_loss/num_batches:.4f}",
                "lr": f"{scheduler.get_last_lr()[0]:.2e}"
            })

    # Handle remaining gradients
    if (step + 1) % accumulation_steps != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    return {
        "train_loss": total_loss / num_batches,
        "train_ner_loss": total_ner_loss / num_batches,
        "train_sentiment_loss": total_sentiment_loss / num_batches,
    }


@torch.no_grad()
def evaluate_model(model, val_loader, device):
    """Evaluate model on validation set."""
    model.eval()

    ner_criterion = nn.CrossEntropyLoss(label_smoothing=0.1, ignore_index=-100)
    sentiment_criterion = nn.MSELoss(reduction="none")

    total_ner_loss = 0.0
    total_sentiment_loss = 0.0
    total_samples = 0
    total_entities = 0

    all_ner_preds = []
    all_ner_labels = []
    all_sentiment_preds = []
    all_sentiment_targets = []

    for batch in tqdm(val_loader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        ner_labels = batch["ner_labels"].to(device)
        entity_masks = batch["entity_masks"].to(device)
        sentiment_targets = batch["sentiment_scores"].to(device)
        entity_mask_valid = batch["entity_mask_valid"].to(device)

        batch_size = input_ids.shape[0]

        with torch.amp.autocast(device_type="cuda"):
            ner_output, sentiment_preds = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                entity_masks=entity_masks,
            )

        # NER loss
        if isinstance(ner_output, dict):
            ner_logits = ner_output["logits"]
            if hasattr(model.ner_head, 'crf'):
                ner_loss = -model.ner_head.crf(
                    ner_logits, ner_labels,
                    mask=attention_mask.bool(), reduction='mean'
                )
            else:
                _, seq_len, num_labels = ner_logits.shape
                ner_loss = ner_criterion(
                    ner_logits.view(-1, num_labels),
                    ner_labels.view(-1),
                )
        else:
            ner_logits = ner_output
            _, seq_len, num_labels = ner_logits.shape
            ner_loss = ner_criterion(
                ner_logits.view(-1, num_labels),
                ner_labels.view(-1),
            )
        total_ner_loss += ner_loss.item() * batch_size

        # Sentiment loss
        sentiment_loss_all = sentiment_criterion(sentiment_preds, sentiment_targets)
        sentiment_loss_masked = sentiment_loss_all * entity_mask_valid
        num_valid = entity_mask_valid.sum().item()
        if num_valid > 0:
            total_sentiment_loss += sentiment_loss_masked.sum().item()
            total_entities += num_valid

        total_samples += batch_size

        # Collect predictions
        if isinstance(ner_output, dict) and "predictions" in ner_output:
            ner_preds = ner_output["predictions"]
        else:
            ner_preds = ner_logits.argmax(dim=-1)

        valid_mask = attention_mask.bool()
        for i in range(batch_size):
            valid_tokens = valid_mask[i]
            all_ner_preds.extend(ner_preds[i][valid_tokens].cpu().tolist())
            all_ner_labels.extend(ner_labels[i][valid_tokens].cpu().tolist())

        for i in range(batch_size):
            for j in range(entity_mask_valid.shape[1]):
                if entity_mask_valid[i, j] > 0:
                    all_sentiment_preds.append(sentiment_preds[i, j].item())
                    all_sentiment_targets.append(sentiment_targets[i, j].item())

    # Compute metrics
    metrics = {
        "val_ner_loss": total_ner_loss / total_samples,
        "val_sentiment_loss": total_sentiment_loss / max(total_entities, 1),
        "val_total_loss": (
            total_ner_loss / total_samples +
            0.5 * total_sentiment_loss / max(total_entities, 1)
        ),
    }

    ner_metrics = compute_ner_metrics(all_ner_preds, all_ner_labels)
    metrics.update(ner_metrics)

    if all_sentiment_preds:
        sentiment_metrics = compute_sentiment_metrics(all_sentiment_preds, all_sentiment_targets)
        metrics.update(sentiment_metrics)

    return metrics

print("Training functions defined.")

## 6. Stage 1: COMPLETED — Load from Checkpoint

Stage 1 NER-only training completed in previous session (5 epochs, ~19 hours).
Best model saved at epoch 5 with val_ner_loss=130.05, NER F1=0.7612.

In [ ]:
# Stage 1 already complete — load results from checkpoint
print("=" * 60)
print("STAGE 1: SKIPPED (loading from checkpoint)")
print("=" * 60)

stage1_ckpt_path = f"{CONFIG['stage1_checkpoint_dir']}/best_model.pt"
assert os.path.exists(stage1_ckpt_path), f"Stage 1 checkpoint not found: {stage1_ckpt_path}"

# Load the last epoch checkpoint to get training history
last_epoch_ckpt = f"{CONFIG['stage1_checkpoint_dir']}/checkpoint_epoch_5.pt"
if os.path.exists(last_epoch_ckpt):
    ckpt = torch.load(last_epoch_ckpt, map_location="cpu", weights_only=False)
    history_stage1 = ckpt.get("history", {"train_loss": [], "val_loss": [], "val_ner_f1": []})
    best_val_loss_s1 = min(history_stage1["val_loss"]) if history_stage1["val_loss"] else float("inf")
    del ckpt
    print(f"\nStage 1 history loaded from checkpoint_epoch_5.pt")
    print(f"  Epochs completed: {len(history_stage1['train_loss'])}")
    print(f"  Best val NER loss: {best_val_loss_s1:.4f}")
    print(f"  Best NER F1: {max(history_stage1['val_ner_f1']):.4f}")
    for i, (tl, vl, f1) in enumerate(zip(
        history_stage1['train_loss'],
        history_stage1['val_loss'],
        history_stage1['val_ner_f1']
    )):
        print(f"  Epoch {i+1}: train_loss={tl:.4f}, val_loss={vl:.4f}, ner_f1={f1:.4f}")
else:
    print("WARNING: Could not find epoch checkpoint for history, using defaults")
    history_stage1 = {"train_loss": [], "val_loss": [], "val_ner_f1": []}
    best_val_loss_s1 = float("inf")

print(f"\nStage 1 best model: {stage1_ckpt_path}")
print("Proceeding directly to Stage 2...")

In [ ]:
# Stage 1 training loop — SKIPPED (already complete)
# Original training ran 5 epochs in ~19 hours
# Results: best val_ner_loss=130.05, best NER F1=0.7612
print("Stage 1 training skipped — checkpoint already saved on Drive")

In [ ]:
# Clear memory between stages
import gc
gc.collect()
torch.cuda.empty_cache()
if torch.cuda.is_available():
    print(f"Memory cleared. GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
else:
    print("Memory cleared.")

## 7. Stage 2: Joint NER + Sentiment Fine-tuning

In [ ]:
print("=" * 60)
print("STAGE 2: JOINT NER + SENTIMENT FINE-TUNING (Longformer-Large)")
print("=" * 60)

# Setup logging for Stage 2
LOG_FILE = f"{PROJECT_PATH}/logs/stage2_large_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
os.makedirs(f"{PROJECT_PATH}/logs", exist_ok=True)

def log_message(msg):
    """Write a message to the log file (append mode)."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(LOG_FILE, "a") as f:
        f.write(f"{timestamp} - {msg}\n")
    print(msg)

log_message("=" * 60)
log_message("STAGE 2: JOINT NER + SENTIMENT FINE-TUNING (Longformer-Large)")
log_message("=" * 60)
log_message(f"Config: batch={CONFIG['batch_size']}x{CONFIG['gradient_accumulation']}, "
            f"max_length={CONFIG['max_length']}, use_crf={CONFIG['use_crf']}")

# Create fresh model and load Stage 1 best weights
model_stage2 = FinancialEntitySentimentModel(
    encoder_name=CONFIG["encoder_name"],
    hidden_size=CONFIG["hidden_size"],
    use_crf_ner=CONFIG["use_crf"],
)
model_stage2 = model_stage2.to(device)

# Load Stage 1 best weights
stage1_ckpt_path = f"{CONFIG['stage1_checkpoint_dir']}/best_model.pt"
log_message(f"Loading Stage 1 weights: {stage1_ckpt_path}")
checkpoint = torch.load(stage1_ckpt_path, map_location=device, weights_only=False)
model_stage2.load_state_dict(checkpoint["model_state_dict"])
log_message(f"Stage 1 weights loaded! (from epoch {checkpoint.get('epoch', -1) + 1})")
del checkpoint
torch.cuda.empty_cache()

# Optimizer & scheduler for Stage 2
optimizer_stage2 = AdamW(model_stage2.parameters(), lr=CONFIG["stage2_lr"], weight_decay=0.01)
num_steps = len(train_loader) * CONFIG["stage2_epochs"]
scheduler_stage2 = CosineAnnealingLR(optimizer_stage2, T_max=num_steps)
scaler_stage2 = torch.amp.GradScaler()

log_message(f"Stage 2: epochs={CONFIG['stage2_epochs']}, lr={CONFIG['stage2_lr']}, steps={num_steps}")
log_message(f"Log file: {LOG_FILE}")

if torch.cuda.is_available():
    log_message(f"GPU: {torch.cuda.get_device_name(0)}")
    log_message(f"GPU Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

In [ ]:
# Train Stage 2 with curriculum learning
log_message("Starting Stage 2 training...")
start_time = time.time()

best_val_loss = float("inf")
history_stage2 = {
    "train_loss": [], "val_loss": [],
    "val_ner_f1": [], "val_sentiment_mse": []
}

accumulation_steps = CONFIG.get("gradient_accumulation", 1)

for epoch in range(CONFIG["stage2_epochs"]):
    epoch_start = time.time()

    # Curriculum learning: adjust weights over epochs
    progress = epoch / max(CONFIG["stage2_epochs"] - 1, 1)
    ner_weight = 1.0 + (0.5 - 1.0) * progress      # 1.0 -> 0.5
    sentiment_weight = 0.3 + (1.0 - 0.3) * progress  # 0.3 -> 1.0

    log_message(f"Epoch {epoch + 1}/{CONFIG['stage2_epochs']} - NER weight: {ner_weight:.2f}, Sentiment weight: {sentiment_weight:.2f}")

    # Train
    train_metrics = train_epoch_with_accumulation(
        model=model_stage2,
        train_loader=train_loader,
        optimizer=optimizer_stage2,
        scheduler=scheduler_stage2,
        scaler=scaler_stage2,
        device=device,
        accumulation_steps=accumulation_steps,
        ner_weight=ner_weight,
        sentiment_weight=sentiment_weight,
    )

    # Evaluate
    val_metrics = evaluate_model(model_stage2, val_loader, device)

    # Log epoch results (this is the only write per epoch - minimal sync)
    epoch_time = time.time() - epoch_start
    gpu_mem = torch.cuda.max_memory_allocated()/1e9 if torch.cuda.is_available() else 0

    log_msg = (
        f"Epoch {epoch + 1} complete: "
        f"train_loss={train_metrics['train_loss']:.4f}, "
        f"val_loss={val_metrics['val_total_loss']:.4f}, "
        f"ner_f1={val_metrics.get('ner_f1', 0):.4f}, "
        f"sent_mse={val_metrics.get('sentiment_mse', 0):.4f}, "
        f"time={epoch_time/60:.1f}min, "
        f"gpu_peak={gpu_mem:.1f}GB"
    )
    log_message(log_msg)

    history_stage2["train_loss"].append(train_metrics["train_loss"])
    history_stage2["val_loss"].append(val_metrics["val_total_loss"])
    history_stage2["val_ner_f1"].append(val_metrics.get("ner_f1", 0))
    history_stage2["val_sentiment_mse"].append(val_metrics.get("sentiment_mse", 0))

    # Save if best
    if val_metrics["val_total_loss"] < best_val_loss:
        best_val_loss = val_metrics["val_total_loss"]
        torch.save({
            "model_state_dict": model_stage2.state_dict(),
            "optimizer_state_dict": optimizer_stage2.state_dict(),
            "val_metrics": val_metrics,
            "epoch": epoch,
        }, f"{CONFIG['stage2_checkpoint_dir']}/best_model.pt")
        log_message(f"  -> New best model saved! (val_loss={best_val_loss:.4f})")

    # Save periodic checkpoint
    torch.save({
        "model_state_dict": model_stage2.state_dict(),
        "optimizer_state_dict": optimizer_stage2.state_dict(),
        "scheduler_state_dict": scheduler_stage2.state_dict(),
        "epoch": epoch,
        "history": history_stage2,
    }, f"{CONFIG['stage2_checkpoint_dir']}/checkpoint_epoch_{epoch+1}.pt")

    # ETA
    elapsed = time.time() - start_time
    time_per_epoch = elapsed / (epoch + 1)
    remaining = time_per_epoch * (CONFIG["stage2_epochs"] - (epoch + 1))
    log_message(f"  -> ETA: {remaining/60:.1f} min remaining")

elapsed = time.time() - start_time
log_message("="*60)
log_message("STAGE 2 COMPLETE!")
log_message(f"Total time: {elapsed/60:.1f} min ({elapsed/3600:.2f} hours)")
log_message(f"Best val loss: {best_val_loss:.4f}")
log_message(f"Best NER F1: {max(history_stage2['val_ner_f1']):.4f}")
log_message(f"Best Sentiment MSE: {min(history_stage2['val_sentiment_mse']):.4f}")
log_message("="*60)

## 8. Training Summary & Visualization

*   List item
*   List item



In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Stage 1: Loss
ax = axes[0, 0]
ax.plot(history_stage1['train_loss'], label='Train')
ax.plot(history_stage1['val_loss'], label='Val')
ax.set_title('Stage 1: NER Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()

# Stage 1: NER F1
ax = axes[0, 1]
ax.plot(history_stage1['val_ner_f1'], 'g-', label='NER F1')
ax.set_title('Stage 1: NER F1')
ax.set_xlabel('Epoch')
ax.set_ylabel('F1 Score')
ax.legend()

# Stage 2: Loss
ax = axes[0, 2]
ax.plot(history_stage2['train_loss'], label='Train')
ax.plot(history_stage2['val_loss'], label='Val')
ax.set_title('Stage 2: Joint Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()

# Stage 2: NER F1
ax = axes[1, 0]
ax.plot(history_stage2['val_ner_f1'], 'g-', label='NER F1')
ax.set_title('Stage 2: NER F1')
ax.set_xlabel('Epoch')
ax.set_ylabel('F1 Score')
ax.legend()

# Stage 2: Sentiment MSE
ax = axes[1, 1]
ax.plot(history_stage2['val_sentiment_mse'], 'r-', label='Sentiment MSE')
ax.set_title('Stage 2: Sentiment MSE')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE')
ax.legend()

# Summary
ax = axes[1, 2]
ax.axis('off')
summary_text = f"""
TRAINING SUMMARY (Longformer-Large)
====================================

Model: {CONFIG['encoder_name']}
Hidden: {CONFIG['hidden_size']}, CRF: {CONFIG['use_crf']}
Batch: {CONFIG['batch_size']} x {CONFIG['gradient_accumulation']} = {CONFIG['batch_size'] * CONFIG['gradient_accumulation']}

Stage 1 (NER only): {CONFIG['stage1_epochs']} epochs
  Best NER F1: {max(history_stage1['val_ner_f1']):.4f}
  Best Val Loss: {best_val_loss_s1:.4f}

Stage 2 (Joint): {CONFIG['stage2_epochs']} epochs
  Best NER F1: {max(history_stage2['val_ner_f1']):.4f}
  Best Sent MSE: {min(history_stage2['val_sentiment_mse']):.4f}
  Best Val Loss: {best_val_loss:.4f}
"""
ax.text(0.05, 0.5, summary_text, fontsize=11, family='monospace',
        verticalalignment='center', transform=ax.transAxes)

plt.tight_layout()
os.makedirs(f"{PROJECT_PATH}/outputs", exist_ok=True)
plt.savefig(f"{PROJECT_PATH}/outputs/training_curves_large.png", dpi=150)
plt.show()
print(f"\nPlot saved to: {PROJECT_PATH}/outputs/training_curves_large.png")

## 9. Final Model Location

In [ ]:
print("=" * 60)
print("TRAINING COMPLETE!")
print("=" * 60)

print(f"\nModel checkpoints:")
print(f"  Stage 1: {CONFIG['stage1_checkpoint_dir']}/best_model.pt")
print(f"  Stage 2: {CONFIG['stage2_checkpoint_dir']}/best_model.pt")

print(f"\nTo use the trained model:")
print(f"""
from models.pipeline import FinancialEntitySentimentModel
import torch

model = FinancialEntitySentimentModel(
    encoder_name="{CONFIG['encoder_name']}",
    hidden_size={CONFIG['hidden_size']},
    use_crf_ner={CONFIG['use_crf']},
)
checkpoint = torch.load("checkpoints/stage2_joint_large/best_model.pt", weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
""")

print(f"\nCheckpoint files:")
!ls -lh "{CONFIG['stage1_checkpoint_dir']}"
print()
!ls -lh "{CONFIG['stage2_checkpoint_dir']}"

## 10. (Optional) Evaluate on Holdout Set

In [ ]:
# Evaluate on holdout set
if os.path.exists(CONFIG["holdout_file"]):
    print("Loading holdout set for evaluation...")

    holdout_loader, _ = create_data_loaders(
        train_files=CONFIG["holdout_file"],
        preprocessor=preprocessor,
        batch_size=CONFIG["batch_size"],
    )

    holdout_metrics = evaluate_model(model_stage2, holdout_loader, device)

    print("\n" + "=" * 60)
    print("HOLDOUT SET EVALUATION (9,371 articles)")
    print("=" * 60)
    print(f"  NER F1:          {holdout_metrics.get('ner_f1', 0):.4f}")
    print(f"  NER Precision:   {holdout_metrics.get('ner_precision', 0):.4f}")
    print(f"  NER Recall:      {holdout_metrics.get('ner_recall', 0):.4f}")
    print(f"  Sentiment MSE:   {holdout_metrics.get('sentiment_mse', 0):.4f}")
    print(f"  Sentiment MAE:   {holdout_metrics.get('sentiment_mae', 0):.4f}")
else:
    print(f"Holdout file not found: {CONFIG['holdout_file']}")

In [ ]:
# Auto-terminate runtime to stop billing
print("All training and evaluation complete. Terminating runtime to save costs...")
from google.colab import runtime
runtime.unassign()